# 11 — Prompt Chaining

In [1]:
import os, warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("API key loaded")


API key loaded


In [2]:
from groq import Groq
client = Groq()
MODEL = "llama-3.1-8b-instant"
MODEL_LARGE = "llama-3.3-70b-versatile"

def chat(messages, model=MODEL, temperature=0.7, max_tokens=512):
    resp = client.chat.completions.create(
        model=model, messages=messages,
        temperature=temperature, max_tokens=max_tokens
    )
    return resp.choices[0].message.content

def system_chat(system, user, **kw):
    return chat([{"role": "system", "content": system},
                 {"role": "user", "content": user}], **kw)

print(f"Groq client ready | default model: {MODEL}")


Groq client ready | default model: llama-3.1-8b-instant


## What is Prompt Chaining?

In [3]:
topic = "The impact of large language models on software engineering"
print(f"Topic: {topic}")
print("Chain: Extract key themes -> Create outline -> Write intro -> Polish intro")


Topic: The impact of large language models on software engineering
Chain: Extract key themes -> Create outline -> Write intro -> Polish intro


In [4]:
step1 = chat([{"role":"user","content":f"List 5 key themes for an essay on: {topic}. Number them 1-5. Be concise."}],
             temperature=0, max_tokens=150)
print("Step 1 - Key themes:")
print(step1.strip())


Step 1 - Key themes:
Here are 5 key themes for an essay on the impact of large language models on software engineering:

1. **Code Generation and Automation**: How large language models can automate code writing, testing, and debugging, and the implications for software development workflows.
2. **Collaboration and Communication**: The role of large language models in facilitating collaboration among developers, and improving communication between humans and machines.
3. **Code Quality and Maintenance**: The potential of large language models to improve code quality, detect bugs, and reduce maintenance costs.
4. **Security and Vulnerability**: The risks and challenges associated with using large language models in software development, including the potential for introducing security vulnerabilities.
5. **Ethics and Bias**: The need for large language models


In [5]:
step2 = chat([{"role":"user","content":f"Using these themes:\n{step1}\n\nCreate a 5-section essay outline for: {topic}\nUse the themes as sections. Give each section a title and one-line description."}],
             temperature=0, max_tokens=200)
print("Step 2 - Outline:")
print(step2.strip())


Step 2 - Outline:
Here's a 5-section essay outline on the impact of large language models on software engineering:

**I. Code Generation and Automation: Revolutionizing Software Development Workflows**
This section will explore how large language models can automate code writing, testing, and debugging, and the implications for software development workflows.

**II. Collaboration and Communication: Enhancing Human-Machine Interaction**
This section will discuss the role of large language models in facilitating collaboration among developers, and improving communication between humans and machines.

**III. Code Quality and Maintenance: Improving Efficiency and Reducing Costs**
This section will examine the potential of large language models to improve code quality, detect bugs, and reduce maintenance costs, leading to more efficient software development processes.

**IV. Security and Vulnerability: Mitigating Risks and Ensuring Trust**
This section will highlight the risks and challenge

In [6]:
step3 = chat([{"role":"user","content":f"Using this outline:\n{step2}\n\nWrite a compelling 3-sentence introduction paragraph for the essay."}],
             temperature=0.5, max_tokens=150)
print("Step 3 - Introduction:")
print(step3.strip())


Step 3 - Introduction:
The integration of large language models into software engineering has the potential to revolutionize the way developers design, build, and maintain software systems. By automating mundane tasks, improving collaboration and communication, and enhancing code quality, large language models can significantly streamline software development workflows, leading to increased efficiency, reduced costs, and improved overall quality. As these models continue to evolve and mature, it is essential to explore their impact on software engineering and identify the opportunities, challenges, and implications for the industry.


In [7]:
step4 = chat([{"role":"user","content":f"Polish this paragraph for a professional audience. Make it more engaging and precise. Return only the improved paragraph:\n\n{step3}"}],
             temperature=0.3, max_tokens=150)
print("Step 4 - Polished introduction:")
print(step4.strip())


Step 4 - Polished introduction:
The integration of large language models is poised to transform software engineering, empowering developers to design, build, and maintain software systems with unprecedented efficiency and precision. By automating routine tasks, fostering seamless collaboration, and elevating code quality, these models can significantly accelerate software development workflows, yielding substantial cost savings and enhanced overall quality. As large language models continue to advance, it is crucial to investigate their far-reaching impact on software engineering, uncovering opportunities, navigating challenges, and informing strategic decisions for the industry.


## Conditional Chaining

In [8]:
review = "The product is decent but the battery life could be better."

sentiment = chat([{"role":"user","content":f"Classify as POSITIVE, NEGATIVE, or MIXED (one word only): {review}"}],
                 temperature=0, max_tokens=5).strip().upper()
print(f"Sentiment: {sentiment}")

if "NEGATIVE" in sentiment:
    response_type = "apologetic and solution-focused"
elif "POSITIVE" in sentiment:
    response_type = "grateful and encouraging"
else:
    response_type = "empathetic and improvement-focused"

response = chat([{"role":"user","content":f"Write a {response_type} 2-sentence customer service reply to: '{review}'"}],
                temperature=0.3, max_tokens=100)
print(f"Response type: {response_type}")
print(f"Reply: {response.strip()}")


Sentiment: MIXED


Response type: empathetic and improvement-focused
Reply: "I appreciate your honest feedback about our product, and I'm sorry to hear that the battery life didn't meet your expectations - we're always looking for ways to improve our products and will take your comment into consideration for future updates."


## Data Transformation Chain

In [9]:
import json

raw_text = "Alice Smith joined the team on March 15 2023. She earns $95000 annually and works in the engineering department in New York."

extract = chat([{"role":"user","content":f"Extract: name, join_date, salary, department, city from:\n{raw_text}\nReturn as key: value pairs."}],
               temperature=0, max_tokens=100)
print("Extracted:")
print(extract.strip())

format_prompt = f"Convert these key-value pairs to valid JSON with snake_case keys:\n{extract}"
formatted = chat([{"role":"user","content":format_prompt}], temperature=0, max_tokens=100)
print("\nFormatted JSON:")
print(formatted.strip())


Extracted:
Here are the extracted key-value pairs:

- name: Alice Smith
- join_date: March 15, 2023
- salary: 95000
- department: engineering
- city: New York



Formatted JSON:
Here's the JSON representation of the given key-value pairs with snake_case keys:

```json
{
  "name": "Alice Smith",
  "join_date": "March 15, 2023",
  "salary": 95000,
  "department": "engineering",
  "city": "New York"
}
```

If you want to create this JSON in a programming language, here's an example in Python:

```python
import json

data = {
    "name":
